# Lesson 7: Models and Choices

**Time:** ~40 minutes | **Cost:** $0 (no API calls)

## Why Different Models?

Different AI models exist for different tasks:

- **Claude Sonnet** — fast, affordable, great with tools and structured output
- **Claude Opus** — slower, more expensive, best for complex reasoning
- **Grok-4** — excellent at long-form writing, different strengths

No single model is "the best" — it depends on the task.

## Speed / Cost / Quality Tradeoffs

Every model balances three factors. You can't have all three at maximum.

| Model | Speed | Cost | Best for |
|-------|-------|------|----------|
| Claude Haiku | Very fast | Very cheap | Simple tasks, classification |
| Claude Sonnet | Fast | Moderate | Tools, structured output, analysis |
| Claude Opus | Slow | Expensive | Complex reasoning, leadership |
| Grok-4 | Medium | Moderate | Long-form writing, creative content |

In [ ]:
# Let's estimate the cost of our pipeline per article

# Approximate costs per 1M tokens (as of 2025)
models = {
    "Claude Sonnet": {"input": 3.00, "output": 15.00},
    "Claude Opus": {"input": 15.00, "output": 75.00},
    "Grok-4": {"input": 3.00, "output": 15.00},
}

# Our pipeline token usage (approximate)
pipeline_steps = [
    {"step": "Research Agent", "model": "Claude Sonnet", "input_tokens": 2000, "output_tokens": 1500},
    {"step": "Outline Agent", "model": "Claude Sonnet", "input_tokens": 3000, "output_tokens": 1000},
    {"step": "Writer Agent", "model": "Grok-4", "input_tokens": 3000, "output_tokens": 3500},
    {"step": "Image Agent", "model": "Claude Sonnet", "input_tokens": 5000, "output_tokens": 500},
]

total_cost = 0
print("Cost breakdown per article:\n")
for step in pipeline_steps:
    model = models[step["model"]]
    input_cost = step["input_tokens"] / 1_000_000 * model["input"]
    output_cost = step["output_tokens"] / 1_000_000 * model["output"]
    step_cost = input_cost + output_cost
    total_cost += step_cost
    print(f"  {step['step']:20s} ({step['model']:14s}): ${step_cost:.4f}")

print(f"\n  {'TOTAL':20s}                  : ${total_cost:.4f}")
print(f"\n  For 100 articles: ${total_cost * 100:.2f}")
print(f"  For 1000 articles: ${total_cost * 1000:.2f}")

## Our Model Choices Explained

The decisions made in `builders.py` — the file that defines all our pipeline agents:

| Agent | Model | Why this model? |
|-------|-------|----------------|
| Research Agent | Claude Sonnet | Needs **tools** (DuckDuckGo). Sonnet is fast and affordable. |
| Outline Agent | Claude Sonnet | Needs **output_schema** (structured JSON). Sonnet handles both tools and schemas. |
| Writer Agent | Grok-4 | Natural, engaging writing. No tools or schema needed — plain Markdown. |
| Image Agent | Claude Sonnet | Needs **both** tools (image APIs) and output_schema. Only Claude supports this combination. |
| Team Leader | Claude Opus | Complex reasoning to understand user requests and delegate to the right team member. |

**The key constraint:** Claude can combine `tools` + `output_schema` in one agent. Grok **cannot**. This single technical limitation shaped our entire architecture. The Writer Agent uses Grok because it excels at writing, and writing doesn't need tools or schemas.

In [ ]:
# Why the architecture looks the way it does

print("=== Model Capabilities ===\n")

capabilities = {
    "Claude Sonnet": {"tools": True, "output_schema": True, "both_together": True},
    "Claude Opus": {"tools": True, "output_schema": True, "both_together": True},
    "Grok-4": {"tools": True, "output_schema": True, "both_together": False},
}

for model, caps in capabilities.items():
    check = lambda x: "Yes" if x else "NO"
    print(f"  {model:15s}  tools: {check(caps['tools']):3s}  schema: {check(caps['output_schema']):3s}  both together: {check(caps['both_together'])}")

print("\n=== Pipeline Agent Requirements ===\n")

agents = [
    {"name": "Research Agent", "needs_tools": True, "needs_schema": False, "needs_both": False, "model": "Claude Sonnet"},
    {"name": "Outline Agent", "needs_tools": False, "needs_schema": True, "needs_both": False, "model": "Claude Sonnet"},
    {"name": "Writer Agent", "needs_tools": False, "needs_schema": False, "needs_both": False, "model": "Grok-4"},
    {"name": "Image Agent", "needs_tools": True, "needs_schema": True, "needs_both": True, "model": "Claude Sonnet"},
]

for agent in agents:
    needs = []
    if agent["needs_tools"]: needs.append("tools")
    if agent["needs_schema"]: needs.append("schema")
    if agent["needs_both"]: needs.append("both together")
    needs_str = ", ".join(needs) if needs else "neither"
    print(f"  {agent['name']:18s} needs: {needs_str:20s} \u2192 {agent['model']}")

print("\nWriter uses Grok because it only needs plain text output.")
print("Image Agent MUST use Claude because it needs tools + schema together.")

## Embeddings — A Different Kind of AI

Not all AI outputs are text. **Embeddings** turn text into lists of numbers that capture meaning.

GPS coordinates are a useful comparison: "Tokyo" and "Japan's capital" are close together on a map of meaning. "Baking bread" is far away.

```
"SEO"                        → [0.82, 0.15, 0.91, ...]
"search engine optimization" → [0.80, 0.17, 0.89, ...]  ← close!
"baking bread"               → [0.12, 0.88, 0.05, ...]  ← far away
```

Where embeddings are used:
- **Semantic search** — find articles similar to a query (not keyword matching)
- **Content clustering** — group similar articles together
- **RAG** (Retrieval-Augmented Generation) — find relevant documents to feed to an LLM

We don't use embeddings in this pipeline, but they're a key concept for future AI tools.

In [ ]:
# Simplified embedding demo (real embeddings use hundreds of dimensions)
# This shows the CONCEPT — real numbers come from embedding models

fake_embeddings = {
    "SEO optimization": [0.9, 0.8, 0.1, 0.2],
    "search engine ranking": [0.85, 0.75, 0.15, 0.25],
    "content marketing": [0.6, 0.5, 0.7, 0.3],
    "baking sourdough bread": [0.1, 0.1, 0.2, 0.9],
}

def similarity(a, b):
    """Simple dot product similarity (real systems use cosine similarity)"""
    return sum(x * y for x, y in zip(a, b))

print("Similarity scores (higher = more similar):\n")
base = "SEO optimization"
base_emb = fake_embeddings[base]

for text, emb in fake_embeddings.items():
    if text != base:
        score = similarity(base_emb, emb)
        bar = "\u2588" * int(score * 10)
        print(f"  \"{base}\" vs \"{text}\"")
        print(f"    Score: {score:.2f}  {bar}\n")

## Choosing the Right Model — Decision Guide

When you're deciding which model to use for an agent, follow this flowchart:

1. **Does the agent need tools** (web search, APIs)?
   - Yes → Must use a model that supports tools (Claude Sonnet/Opus)
2. **Does the agent need structured output** (output_schema)?
   - Yes → Must use a model that supports output_schema
3. **Does it need BOTH tools AND structured output?**
   - Yes → **Must use Claude** (Grok can't do both)
   - No → More flexibility in model choice
4. **Is the task primarily long-form writing?**
   - Yes → Consider Grok (great writing quality)
5. **Is the task complex reasoning/delegation?**
   - Yes → Consider Opus (most capable, but expensive)
6. **Default** → Claude Sonnet (best balance of speed, cost, capability)

In [ ]:
# Exercise: Match agents to models!
# For each scenario, which model would you choose and why?

scenarios = [
    {
        "name": "Email Writer Agent",
        "description": "Writes marketing emails from a brief",
        "needs_tools": False,
        "needs_schema": False,
        "priority": "Writing quality",
    },
    {
        "name": "Competitor Analyzer",
        "description": "Searches the web and returns structured competitor data",
        "needs_tools": True,
        "needs_schema": True,
        "priority": "Accuracy",
    },
    {
        "name": "Content Classifier",
        "description": "Categorizes articles into topics (returns a category label)",
        "needs_tools": False,
        "needs_schema": True,
        "priority": "Speed and cost",
    },
]

print("=== Model Selection Exercise ===\n")
for s in scenarios:
    print(f"Agent: {s['name']}")
    print(f"  Task: {s['description']}")
    print(f"  Needs tools: {'Yes' if s['needs_tools'] else 'No'}")
    print(f"  Needs schema: {'Yes' if s['needs_schema'] else 'No'}")
    print(f"  Priority: {s['priority']}")
    print(f"  Your choice: _______________")
    print()

print("Think about your answers, then check the suggested answers below!")

In [ ]:
# Suggested answers:

answers = {
    "Email Writer Agent": "Grok-4 \u2014 writing quality is the priority, no tools or schema needed",
    "Competitor Analyzer": "Claude Sonnet \u2014 needs both tools AND schema together, only Claude supports this",
    "Content Classifier": "Claude Haiku \u2014 simple task, schema needed but it's lightweight, speed and cost matter most",
}

print("=== Suggested Answers ===\n")
for agent, answer in answers.items():
    print(f"  {agent}: {answer}\n")

## Module 2 Summary — Understanding AI

| Lesson | Topic | Key takeaway |
|--------|-------|-------------|
| Lesson 5 | How LLMs Work | LLMs predict text token by token. They have knowledge cutoffs and can hallucinate. |
| Lesson 6 | Prompts and Context | Good prompts = Role + Task + Constraints + Examples. `instructions` is the system prompt. |
| Lesson 7 | Models and Choices | Different models for different tasks. Architecture is shaped by model capabilities. |

You now understand **WHY** agents work the way they do. Module 3 is where you **BUILD** them.

**Next module:** Module 3 — Building Agents. You'll create your first AI agent, give it tools, make it return structured data, chain agents together, and build a mini pipeline.

## Exercise

No code needed — thought exercise:

Your SEO agency wants to build a new AI tool: a **"Content Audit Agent"** that:
- Crawls a website URL to read existing content (needs a tool)
- Analyzes the content for SEO issues
- Returns a structured report with scores and recommendations (needs schema)
- Must be affordable to run on hundreds of pages

**Questions:**
1. Which model would you choose? Why?
2. Could you use Grok for this? Why or why not?
3. How would you keep costs down when processing hundreds of pages?

<details>
<summary>Click to reveal suggested answers</summary>

1. **Claude Sonnet** — it needs both tools (web crawling) and structured output (report schema), and Sonnet is affordable enough for bulk use.
2. **No** — Grok cannot combine tools and output_schema in one agent. You'd need to split it into two separate agents (one for crawling, one for reporting), which adds complexity.
3. Use **Claude Haiku** for a pre-filtering step (quick check if a page even needs a full audit), then only send pages that need deep analysis to Sonnet. Also, keep prompts concise to minimize input tokens.

</details>

## Ready for Module 3? — Checkpoint

Module 3 is where it gets real: you'll create actual AI agents that call Claude and Grok APIs.

- **Real API calls** — each cell costs a small amount (~$0.01-0.10)
- **You need API keys** — `ANTHROPIC_API_KEY` and `XAI_API_KEY` must be set in your `.env` file
- **Internet required** — agents call external APIs

Run the cell below to verify you're ready. **Both checks must pass.**

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

checks_passed = 0

# Check 1: Anthropic API key
print("1. Anthropic API key (for Claude)...")
key = os.getenv("ANTHROPIC_API_KEY", "")
if len(key) > 10:
    print(f"   PASS - Key found ({key[:8]}...)")
    checks_passed += 1
else:
    print("   FAIL - ANTHROPIC_API_KEY not set")
    print("   Fix: Add your key to the .env file in the project root")
    print("   Get one at: https://console.anthropic.com")

# Check 2: xAI API key
print("2. xAI API key (for Grok)...")
key = os.getenv("XAI_API_KEY", "")
if len(key) > 5:
    print(f"   PASS - Key found ({key[:8]}...)")
    checks_passed += 1
else:
    print("   FAIL - XAI_API_KEY not set")
    print("   Fix: Add your key to the .env file in the project root")
    print("   Get one at: https://console.x.ai")

print()
if checks_passed == 2:
    print("You're ready for Module 3!")
else:
    print(f"Fix the failed check(s) before starting Module 3.")
    print("Module 3 notebooks will fail without valid API keys.")